In [9]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.nn import GATConv  # GATConvをインポート

# ----------------------------
# モデル定義 (GAT Model)
# ----------------------------
class GATModel(torch.nn.Module):
    def __init__(self, hidden_channels=64, num_classes=10):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=4, concat=True)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=4, concat=True)
        self.conv3 = GATConv(hidden_channels * 8, hidden_channels, heads=4, concat=True)
        self.conv4 = GATConv(hidden_channels * 4, hidden_channels, heads=4, concat=True)
        self.fc = nn.Linear(hidden_channels * 4, num_classes)
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv4(x, edge_index))
        x = self.fc(x)
        return x

# ----------------------------
# データペアリング関数 (prepare_data)
# ----------------------------
def prepare_data(pairs, standardized_data_folder, label_data_folder, x_coords, y_coords, z_coords, edge_index):
    data_list = []
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]

        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        x = torch.tensor(node_features, dtype=torch.float)

        y = torch.argmax(torch.tensor(label, dtype=torch.float), dim=1).long()

        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
    return data_list

# ----------------------------
# モデルのロードと予測 (load and predict)
# ----------------------------
def load_and_predict(model_path, test_loader, device):
    # モデルをロード
    model = GATModel(hidden_channels=64, num_classes=10).to(device)  # GATModelを使用
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # 予測結果と正解ラベルを保存
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            pred = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(pred)
            all_labels.extend(batch.y.cpu().numpy())

    # NumPy 配列に変換
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    return all_preds, all_labels

# ----------------------------
# 可視化関数 (Visualization functions)
# ----------------------------
def plot_confusion_matrix(all_labels, all_preds, class_names):
    # 混同行列を作成
    cm = confusion_matrix(all_labels, all_preds, normalize='true')

    # 混同行列を表示
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.ylabel('True Label', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.title('Normalized Confusion Matrix', fontsize=16)
    plt.show()

def plot_comparison(all_labels, all_preds):
    # 正解ラベルと予測ラベルの比較を散布図で可視化
    correct = all_labels == all_preds
    incorrect = ~correct
    plt.figure(figsize=(12, 6))
    plt.scatter(np.where(correct)[0], all_labels[correct], label='Correct', color='blue', alpha=0.6)
    plt.scatter(np.where(incorrect)[0], all_labels[incorrect], label='Incorrect', color='red', alpha=0.6)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Label')
    plt.title('Comparison of True Labels and Predictions')
    plt.show()

# クラス別の評価指標を表示
def print_classification_report(all_labels, all_preds, class_names):
    report = classification_report(all_labels, all_preds, target_names=class_names)
    print(report)

# ----------------------------
# メインプログラム (Main program)
# ----------------------------
if __name__ == '__main__':
    # デバイス設定
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # モデルのパス
    model_path = "/home/nishioka/GNN/GNNmodel/10classmodel/GATModel_20241022_111748best_model_fold_1.pth"  # GATModel用

    # テストデータの準備
    standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
    label_data_folder = "/home/nishioka/GNN/Defect10Class_OneHot_test1"
    x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
    y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
    z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]
    edge_index = torch.tensor(np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy").T, dtype=torch.long).to(device)

    # データ準備 (train_test_splitを使用してデータを分割)
    data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
    label_files = [f for f in os.listdir(label_data_folder) if f.startswith("Defect10Class_L")]

    # データペアを作成
    pairs = [(data_files[i], label_files[i]) for i in range(len(data_files))]
    
    # データをトレーニング、テストに分割
    train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

    test_dataset = prepare_data(test_pairs, standardized_data_folder, label_data_folder, x_coords, y_coords, z_coords, edge_index)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # 予測
    all_preds, all_labels = load_and_predict(model_path, test_loader, device)

    # 混同行列の可視化
    class_names = [f'Class {i}' for i in range(10)]  # クラス名
    plot_confusion_matrix(all_labels, all_preds, class_names)

    # 正解と予測の比較
    plot_comparison(all_labels, all_preds)

    # クラス別の評価指標を表示
    print_classification_report(all_labels, all_preds, class_names)


OutOfMemoryError: CUDA out of memory. Tried to allocate 674.00 MiB. GPU 0 has a total capacity of 10.75 GiB of which 86.50 MiB is free. Process 1943114 has 2.44 GiB memory in use. Including non-PyTorch memory, this process has 8.23 GiB memory in use. Of the allocated memory 7.51 GiB is allocated by PyTorch, and 550.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)